In [ ]:
import openvino as ov

core = ov.Core()
print(core.available_devices)

from pathlib import Path
from ovomodel import OVOBertModel

model_path = Path("./outputs/intel/bert_base_uncased_scl/model/model.onnx").resolve()
bert = OVOBertModel(model_path)
bert.model.session.get_providers()

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("intel/bert-base-uncased-mrpc")

def tokenize(s1, s2):
    return tokenizer(
        *[s1, s2],
        padding="max_length",
        max_length=512,
        truncation=True,
        add_special_tokens=True,
        return_tensors="pt"
    )

In [ ]:
encoded_inputs = tokenize("Hello", "World")
for _ in range(100):
    model.forward(**encoded_inputs)

In [ ]:
import openvino as ov

core = ov.Core()
print(core.available_devices)

from pathlib import Path

model_path = Path("./models/mlm/bert/outputs/intel/bert_base_uncased_scl/model/model.onnx").resolve()
print(f"Model path: {model_path}")

import onnxruntime as ort

options = ort.SessionOptions()
options.add_session_config_entry('session.disable_cpu_ep_fallback', '0')
options.add_session_config_entry('ep.context_enable', '1')
options.add_session_config_entry('ep.context_embed_mode', '0')
options.add_session_config_entry('ep.context_file_path', str(model_path.with_suffix('.onnx_ctx.onnx')))

session = ort.InferenceSession(
    model_path,
    sess_options=options,
    providers=['OpenVINOExecutionProvider'],
    provider_options=[{
        'device_type': 'NPU',
        'enable_qdq_optimizer': True,
        'cache_dir': str(model_path.parent),
    }],
)
session.get_providers()